In [0]:
%sql
SELECT current_catalog(), current_schema();

SHOW CATALOGS;

catalog
dbacademy
samples
system
workspace


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.migration_dev;
CREATE SCHEMA IF NOT EXISTS workspace.migration_test;
CREATE SCHEMA IF NOT EXISTS workspace.migration_prod_sim;

SHOW SCHEMAS IN workspace;

databaseName
default
information_schema
migration_dev
migration_prod_sim
migration_test


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.migration_dev.landing;
CREATE VOLUME IF NOT EXISTS workspace.migration_dev.checkpoints;
CREATE VOLUME IF NOT EXISTS workspace.migration_dev.archive;

SHOW VOLUMES IN workspace.migration_dev;

database,volume_name
migration_dev,archive
migration_dev,checkpoints
migration_dev,landing


In [0]:
%python
base_path = "/Volumes/workspace/migration_dev/landing"

dbutils.fs.mkdirs(f"{base_path}/batch_001/pdfs")
dbutils.fs.mkdirs(f"{base_path}/batch_001/metadata")

display(dbutils.fs.ls(f"{base_path}/batch_001"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/migration_dev/landing/batch_001/metadata/,metadata/,0,1791901349120
dbfs:/Volumes/workspace/migration_dev/landing/batch_001/pdfs/,pdfs/,0,1791901349120


In [0]:
%pip install reportlab

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 39.7 MB/s  0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
%python
from pathlib import Path
from io import BytesIO
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4

pdf_dir = Path(
    "/Volumes/workspace/migration_dev/landing/batch_001/pdfs"
)
pdf_dir.mkdir(parents=True, exist_ok=True)

created = 0

for i in range(1, 101):
    document_id = f"DOC{i:06d}"
    account_number = f"TEST{i:08d}"
    customer_name = f"Sample Customer {i:03d}"
    pdf_path = pdf_dir / f"{document_id}.pdf"

    # Preserve existing files when this cell is rerun.
    if pdf_path.exists():
        continue

    buffer = BytesIO()
    pdf = canvas.Canvas(buffer, pagesize=A4)
    pdf.setFont("Helvetica", 12)

    lines = [
        "SYNTHETIC DATA — TRAINING ONLY",
        "Document type: Account statement",
        "Batch ID: batch_001",
        f"Document ID: {document_id}",
        f"Account number: {account_number}",
        f"Customer name: {customer_name}",
        "Statement date: 2026-09-30",
    ]

    for row, text in enumerate(lines):
        pdf.drawString(50, 790 - row * 25, text)

    pdf.showPage()
    pdf.save()

    with pdf_path.open("wb") as output:
        output.write(buffer.getvalue())

    created += 1

print(f"New PDFs created: {created}")
print(f"Total PDFs in batch: {len(list(pdf_dir.glob('*.pdf')))}")

New PDFs created: 100
Total PDFs in batch: 100


In [0]:
%python
import csv
import hashlib
from pathlib import Path

batch_path = Path(
    "/Volumes/workspace/migration_dev/landing/batch_001"
)
csv_path = batch_path / "metadata" / "statements.csv"

columns = [
    "batch_id",
    "document_id",
    "account_number",
    "customer_name",
    "document_type",
    "statement_date",
    "file_name",
    "pdf_path",
    "source_sha256",
]

rows = []

for i in range(1, 101):
    document_id = f"DOC{i:06d}"
    pdf_path = batch_path / "pdfs" / f"{document_id}.pdf"

    # Stop if an expected source PDF is missing.
    if not pdf_path.is_file():
        raise FileNotFoundError(f"Missing PDF: {pdf_path}")

    rows.append({
        "batch_id": "batch_001",
        "document_id": document_id,
        "account_number": f"TEST{i:08d}",
        "customer_name": f"Sample Customer {i:03d}",
        "document_type": "ACCOUNT_STATEMENT",
        "statement_date": "2026-09-30",
        "file_name": pdf_path.name,
        "pdf_path": str(pdf_path),
        "source_sha256": hashlib.sha256(
            pdf_path.read_bytes()
        ).hexdigest(),
    })

# Preserve the source CSV on subsequent runs.
if csv_path.exists():
    print("CSV already exists; preserved existing file.")
else:
    with csv_path.open("w", newline="", encoding="utf-8") as output:
        writer = csv.DictWriter(output, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)
    print("CSV created.")

with csv_path.open(newline="", encoding="utf-8") as source:
    saved_rows = list(csv.DictReader(source))

print(f"Metadata records: {len(saved_rows)}")
print(f"CSV path: {csv_path}")
print(f"First document: {saved_rows[0]['document_id']}")

CSV created.
Metadata records: 100
CSV path: /Volumes/workspace/migration_dev/landing/batch_001/metadata/statements.csv
First document: DOC000001


In [0]:
%python
import csv
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

batch_path = Path(
    "/Volumes/workspace/migration_dev/landing/batch_001"
)
csv_path = batch_path / "metadata" / "statements.csv"
manifest_path = batch_path / "manifest.json"

with csv_path.open(newline="", encoding="utf-8") as source:
    records = list(csv.DictReader(source))

expected_files = [row["file_name"] for row in records]
actual_files = {p.name for p in (batch_path / "pdfs").glob("*.pdf")}

assert len(records) == 100, "Expected 100 metadata records"
assert len(set(expected_files)) == 100, "Duplicate filenames in CSV"
assert set(expected_files) == actual_files, "PDF and CSV file sets differ"

manifest = {
    "batch_id": "batch_001",
    "schema_version": 1,
    "status": "READY",
    "expected_document_count": 100,
    "expected_metadata_count": 100,
    "metadata_path": str(csv_path),
    "metadata_sha256": hashlib.sha256(
        csv_path.read_bytes()
    ).hexdigest(),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

if manifest_path.exists():
    print("Existing manifest preserved.")
else:
    with manifest_path.open("x", encoding="utf-8") as output:
        json.dump(manifest, output, indent=2)
    print("Source batch marked READY.")

print(manifest_path.read_text(encoding="utf-8"))

Source batch marked READY.
{
  "batch_id": "batch_001",
  "schema_version": 1,
  "status": "READY",
  "expected_document_count": 100,
  "expected_metadata_count": 100,
  "metadata_path": "/Volumes/workspace/migration_dev/landing/batch_001/metadata/statements.csv",
  "metadata_sha256": "010d36296c0af14fea5e461828ceb8173569bcaac82d0ca43d95f3ce0bbbb327",
  "created_at_utc": "2026-10-13T14:43:58.625646+00:00"
}


In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.migration_dev.bronze_metadata (
    batch_id STRING,
    document_id STRING,
    account_number STRING,
    customer_name STRING,
    document_type STRING,
    statement_date STRING,
    file_name STRING,
    pdf_path STRING,
    source_sha256 STRING,
    _rescued_data STRING,
    source_file STRING,
    source_modification_time TIMESTAMP,
    ingestion_timestamp TIMESTAMP
)
USING DELTA
COMMENT 'Raw document metadata with source tracking and ingestion audit fields';

DESCRIBE TABLE workspace.migration_dev.bronze_metadata;

col_name,data_type,comment
batch_id,string,null
document_id,string,null
account_number,string,null
customer_name,string,null
document_type,string,null
statement_date,string,null
file_name,string,null
pdf_path,string,null
source_sha256,string,null
_rescued_data,string,null


In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.migration_dev.validation_results (
    validation_run_id STRING,
    batch_id STRING,
    check_name STRING,
    failures BIGINT,
    result STRING,
    checked_at TIMESTAMP
)
USING DELTA;

CREATE TABLE IF NOT EXISTS workspace.migration_dev.batch_control (
    batch_id STRING,
    validation_run_id STRING,
    batch_status STRING,
    expected_document_count BIGINT,
    actual_document_count BIGINT,
    expected_metadata_count BIGINT,
    actual_metadata_count BIGINT,
    failed_checks BIGINT,
    updated_at TIMESTAMP
)
USING DELTA;

SHOW TABLES IN workspace.migration_dev;

database,tableName,isTemporary
migration_dev,batch_control,false
migration_dev,bronze_file_inventory,false
migration_dev,bronze_metadata,false
migration_dev,validation_results,false


In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.migration_dev.silver_documents (
    batch_id STRING,
    document_id STRING,
    account_number STRING,
    customer_name STRING,
    document_type STRING,
    statement_date DATE,
    file_name STRING,
    pdf_path STRING,
    source_sha256 STRING,
    file_size_bytes BIGINT,
    validation_run_id STRING,
    processed_at TIMESTAMP
)
USING DELTA
COMMENT 'Standardized document metadata from approved batches';

DESCRIBE TABLE workspace.migration_dev.silver_documents;

col_name,data_type,comment
batch_id,string,null
document_id,string,null
account_number,string,null
customer_name,string,null
document_type,string,null
statement_date,date,null
file_name,string,null
pdf_path,string,null
source_sha256,string,null
file_size_bytes,bigint,null


In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.migration_dev.gold_load_manifest (
    batch_id STRING,
    document_id STRING,
    account_number STRING,
    document_type STRING,
    statement_date DATE,
    file_name STRING,
    pdf_path STRING,
    source_sha256 STRING,
    file_size_bytes BIGINT,
    validation_run_id STRING,
    published_at TIMESTAMP
)
USING DELTA
COMMENT 'Approved document loading instructions for the downstream loader';

DESCRIBE TABLE workspace.migration_dev.gold_load_manifest;

col_name,data_type,comment
batch_id,string,null
document_id,string,null
account_number,string,null
document_type,string,null
statement_date,date,null
file_name,string,null
pdf_path,string,null
source_sha256,string,null
file_size_bytes,bigint,null
validation_run_id,string,null


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.migration_dev.downstream;

CREATE TABLE IF NOT EXISTS workspace.migration_dev.load_acknowledgements (
    batch_id STRING,
    document_id STRING,
    validation_run_id STRING,
    load_status STRING,
    destination_path STRING,
    destination_sha256 STRING,
    error_message STRING,
    loaded_at TIMESTAMP
)
USING DELTA
COMMENT 'Latest downstream load outcome per batch document';

DESCRIBE TABLE workspace.migration_dev.load_acknowledgements;

col_name,data_type,comment
batch_id,string,null
document_id,string,null
validation_run_id,string,null
load_status,string,null
destination_path,string,null
destination_sha256,string,null
error_message,string,null
loaded_at,timestamp,null


In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.migration_dev.downstream_reconciliation (
    reconciliation_run_id STRING,
    batch_id STRING,
    validation_run_id STRING,
    expected_document_count BIGINT,
    manifest_document_count BIGINT,
    acknowledgement_count BIGINT,
    reconciled_document_count BIGINT,
    unresolved_document_count BIGINT,
    load_completion_status STRING,
    reconciled_at TIMESTAMP
)
USING DELTA
COMMENT 'Historical downstream reconciliation outcomes per batch';

DESCRIBE TABLE workspace.migration_dev.downstream_reconciliation;

col_name,data_type,comment
reconciliation_run_id,string,null
batch_id,string,null
validation_run_id,string,null
expected_document_count,bigint,null
manifest_document_count,bigint,null
acknowledgement_count,bigint,null
reconciled_document_count,bigint,null
unresolved_document_count,bigint,null
load_completion_status,string,null
reconciled_at,timestamp,null


In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.migration_dev.document_registry (
    batch_id STRING,
    document_id STRING,
    account_number STRING,
    document_type STRING,
    statement_date DATE,
    archive_path STRING,
    document_sha256 STRING,
    file_size_bytes BIGINT,
    validation_run_id STRING,
    reconciliation_run_id STRING,
    archived_at TIMESTAMP,
    retention_until DATE,
    legal_hold BOOLEAN
)
USING DELTA
COMMENT 'Archived document metadata for retrieval and retention tracking';

DESCRIBE TABLE workspace.migration_dev.document_registry;

col_name,data_type,comment
batch_id,string,null
document_id,string,null
account_number,string,null
document_type,string,null
statement_date,date,null
archive_path,string,null
document_sha256,string,null
file_size_bytes,bigint,null
validation_run_id,string,null
reconciliation_run_id,string,null
